# 13 · Hybrid warning tiers (Phase M5)
Final tier = max(ML tier, IMD-rainfall tier adjusted for susceptibility).

In [1]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

In [2]:
from src.warning_levels import run
run()
pd.DataFrame(json.loads(Path('results/warning_levels/warning_thresholds.json').read_text())['alert_level']).T

                      precision  recall     f1  false_alarm_rate  alerts
validation_tier_ml        0.182   0.146  0.162            0.0127   308.0
validation_tier_rain      0.012   0.070  0.021            0.1109  2234.0
validation_tier           0.033   0.216  0.057            0.1236  2542.0
test_tier_ml              0.082   0.275  0.127            0.0602   802.0
test_tier_rain            0.380   0.250  0.302            0.0080   158.0
test_tier                 0.119   0.471  0.191            0.0681   946.0
{'NORMAL': 336664, 'WATCH': 21132, 'ALERT': 24611, 'SEVERE': 7560, 'EXTREME': 1433} {'none': 336664, 'rainfall': 27645, 'ml': 27091}


,precision,recall,f1,false_alarm_rate,alerts
validation_tier_ml,0.182,0.146,0.162,0.0127,308.0
validation_tier_rain,0.012,0.070,0.021,0.1109,2234.0
validation_tier,0.033,0.216,0.057,0.1236,2542.0
test_tier_ml,0.082,0.275,0.127,0.0602,802.0
test_tier_rain,0.380,0.250,0.302,0.0080,158.0
test_tier,0.119,0.471,0.191,0.0681,946.0


## Sanity check on documented disasters (unlabelled period)

In [3]:
r = pd.read_csv('results/warning_levels/replay_warning_levels.csv', parse_dates=['datetime'])
cases = {'Kedarnath route cloudburst': ('2024-07-31', ['Rudraprayag', 'Chamoli', 'Tehri Garhwal']),
         'Dharali / Uttarkashi': ('2025-08-05', ['Uttarkashi', 'Chamoli', 'Rudraprayag'])}
for name, (day, dists) in cases.items():
    d = pd.Timestamp(day)
    w = r[(r.datetime >= d - pd.Timedelta(days=1)) & (r.datetime < d + pd.Timedelta(days=1)) & r.district.isin(dists)]
    display(Markdown(f'**{name}** ({day})'))
    display(w.groupby('station').agg(max_tier=('tier', lambda x: max(x, key=['NORMAL','WATCH','ALERT','SEVERE','EXTREME'].index)),
                                     trigger=('trigger', lambda x: x.value_counts().index[0]), max_p=('p_calibrated', 'max')))

**Kedarnath route cloudburst** (2024-07-31)

,max_tier,trigger,max_p
station,,,
Badrinath_1,NORMAL,none,0.015243
Basukedar,SEVERE,rainfall,0.015243
Gairsain (Ramganga),WATCH,none,0.015243
Guptkashi_1,NORMAL,none,0.015243
Kempty,WATCH,none,0.015243
Sonprayag_1,SEVERE,none,0.015243


**Dharali / Uttarkashi** (2025-08-05)

,max_tier,trigger,max_p
station,,,
Basukedar,NORMAL,none,0.015243
Guptkashi_1,WATCH,rainfall,0.015243
Joshiyara,SEVERE,ml,0.227273
Nandprayag (Nandakini),NORMAL,none,0.015243


## External validation: curated 2024-2025 disasters (independent of training labels)

In [4]:
from src.external_validation import run as ext_run
ext_run()

   event_id       date          district                                             place  stations_reporting hybrid_alert         first_alert  lead_h_to_day_end hybrid_severe ml_alert ml_severe rain_alert rain_severe max_tier trigger_at_max
SUP-2024-04 2024-07-08         Champawat                                 Banbasa, Tanakpur                   2         True 2024-07-07 00:00:00               48.0          True    False     False       True        True  EXTREME       rainfall
SUP-2024-05 2024-07-08 Udham Singh Nagar                                Khatima, Sitarganj                   4        False                None                NaN         False    False     False      False       False    WATCH       rainfall
SUP-2024-01 2024-07-31       Rudraprayag                    Bhimbali, Kedarnath trek route                   3         True 2024-07-30 04:00:00               44.0          True    False     False       True        True   SEVERE       rainfall
SUP-2024-02 2024-07-31     T

,event_id,date,district,place,stations_reporting,hybrid_alert,first_alert,lead_h_to_day_end,hybrid_severe,ml_alert,ml_severe,rain_alert,rain_severe,max_tier,trigger_at_max,source_url
0,SUP-2024-04,2024-07-08,Champawat,"Banbasa, Tanakpur",2,True,2024-07-07 00:00:00,48.0,True,False,False,True,True,EXTREME,rainfall,https://www.newsonair.gov.in/imd-issues-warnin...
1,SUP-2024-05,2024-07-08,Udham Singh Nagar,"Khatima, Sitarganj",4,False,None,NaN,False,False,False,False,False,WATCH,rainfall,https://www.newsonair.gov.in/imd-issues-warnin...
2,SUP-2024-01,2024-07-31,Rudraprayag,"Bhimbali, Kedarnath trek route",3,True,2024-07-30 04:00:00,44.0,True,False,False,True,True,SEVERE,rainfall,https://www.businesstoday.in/india/story/kedar...
3,SUP-2024-02,2024-07-31,Tehri Garhwal,"Ghansali, Nautar stream",1,False,None,NaN,False,False,False,False,False,WATCH,rainfall,https://www.socialnews.xyz/2024/07/31/cloudbur...
4,SUP-2024-03,2024-07-31,Haridwar,"Bhupatwala, Kankhal, Jwalapur",2,True,2024-07-31 20:00:00,4.0,False,False,False,True,False,ALERT,rainfall,https://reliefweb.int/report/india/situation-r...
5,SUP-2025-01,2025-06-29,Uttarkashi,Yamunotri highway (Silai band),1,True,2025-06-28 00:00:00,48.0,True,True,True,False,False,SEVERE,ml,https://news24online.com/india/from-kedarnath-...
6,SUP-2025-02,2025-08-05,Uttarkashi,"Dharali (Kheer Ganga), near Harsil",1,True,2025-08-04 00:00:00,48.0,True,True,True,True,True,SEVERE,ml,https://en.wikipedia.org/wiki/2025_Uttarakhand...
7,SUP-2025-03,2025-08-22,Chamoli,Tharali (Tunri stream),1,False,None,NaN,False,False,False,False,False,NORMAL,none,https://www.newsonair.gov.in/uttarakhand-cloud...
8,SUP-2025-04,2025-08-29,Rudraprayag,multiple villages,0,None,None,NaN,None,None,None,None,None,NO DATA,None,https://www.newsonair.gov.in/uttarakhand-cloud...
9,SUP-2025-05,2025-08-29,Chamoli,Dewal block,1,False,None,NaN,False,False,False,False,False,NORMAL,none,https://www.theweek.in/news/india/2025/08/29/u...
